# 50-modeling-v2 — Prophet + residual stacking (contestant-style)

Pipeline:
- Load `sales.csv` / `sample_submission.csv` only (no parquet FE).
- Calendar + sale-window features (EDA-driven peak months + promotion seasons).
- **Prophet** on `log1p(Revenue)` per CV fold; gradient boosting learns **residual** in log space.
- **XGBoost / LightGBM / CatBoost** with **Huber** (robust) loss; **OOF stacking** with `HuberRegressor` meta-learner.
- **COGS** via `COGS_Ratio = COGS/Revenue` with the same base+meta stack (not a single global ratio).

SHAP: TreeExplainer on the three **residual** models (12 static features).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.chdir('/content/drive/MyDrive/DATATHON2026-GRIDBREAKER/notebooks')

In [3]:
%%capture
!pip install catboost optuna

In [4]:
# ===========================================================================
# 40-modeling-v2 | Prophet residual + Huber GBMs + Huber meta-stack + COGS ratio
# ===========================================================================
import os
import sys
import json
import warnings

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import HuberRegressor
from sklearn.model_selection import GridSearchCV

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from prophet import Prophet

try:
    import shap
    SHAP_OK = True
except ImportError:
    shap = None
    SHAP_OK = False

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
except ImportError:
    optuna = None
    OPTUNA_OK = False

SEED = 42
np.random.seed(SEED)

SUB_DIR = '../submissions'
MODEL_DIR = '../models'
CHART_DIR = '../reports/charts'
for d in (SUB_DIR, MODEL_DIR, CHART_DIR):
    os.makedirs(d, exist_ok=True)

DATA_CANDIDATES = [
    '../data/data_cleaned/sales.csv',
    '../data/sales.csv',
]
SUB_CANDIDATES = [
    '../data/data_cleaned/sample_submission.csv',
    '../data/sample_submission.csv',
]
PARQUET_TRAIN = '../data/features/train_features.parquet'
PARQUET_TEST  = '../data/features/test_features.parquet'


def resolve_first(paths, label):
    for p in paths:
        if os.path.isfile(p):
            return p
    raise FileNotFoundError(f'No {label} found; tried: {paths}')


SALES_PATH = resolve_first(DATA_CANDIDATES, 'sales.csv')
SUB_PATH = resolve_first(SUB_CANDIDATES, 'sample_submission.csv')


def evaluate(y_true, y_pred, prefix=''):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    ss_res = float(np.sum((y_true - y_pred) ** 2))
    ss_tot = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float('nan')
    print(f'  {prefix}MAE={mae:,.0f}  RMSE={rmse:,.0f}  R2={r2:.4f}')
    return {'mae': mae, 'rmse': rmse, 'r2': r2}


print('Setup OK | SEED =', SEED)
print('Parquet available:', os.path.isfile(PARQUET_TRAIN) and os.path.isfile(PARQUET_TEST))
print('Sales CSV fallback:', SALES_PATH)

Setup OK | SEED = 42
Parquet available: True
Sales CSV fallback: ../data/data_cleaned/sales.csv


In [5]:
# --- Sale seasons + calendar features (same idea as high-LB notebook) ---
SALE_SEASONS = [
    {'month': 1,  'start_day': 30, 'duration': 30, 'profit_rank': 1},
    {'month': 3,  'start_day': 18, 'duration': 30, 'profit_rank': 2},
    {'month': 6,  'start_day': 23, 'duration': 29, 'profit_rank': 3},
    {'month': 7,  'start_day': 30, 'duration': 34, 'profit_rank': 5},
    {'month': 8,  'start_day': 30, 'duration': 32, 'profit_rank': 4},
    {'month': 11, 'start_day': 18, 'duration': 45, 'profit_rank': 6},
]


def add_seasonality_features(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df['year'] = df['Date'].dt.year
    df['month'] = df['Date'].dt.month
    df['day'] = df['Date'].dt.day
    df['dayofweek'] = df['Date'].dt.dayofweek
    df['is_peak_season'] = df['Date'].dt.month.isin([4, 5, 11]).astype(int)

    def days_to_nearest_peak(date):
        year = date.year
        candidates = [
            pd.Timestamp(year, 4, 1), pd.Timestamp(year, 5, 1), pd.Timestamp(year, 11, 1),
            pd.Timestamp(year - 1, 4, 1), pd.Timestamp(year - 1, 5, 1), pd.Timestamp(year - 1, 11, 1),
            pd.Timestamp(year + 1, 4, 1), pd.Timestamp(year + 1, 5, 1), pd.Timestamp(year + 1, 11, 1),
        ]
        return min(abs((date - c).days) for c in candidates)

    days_to_peak = df['Date'].map(days_to_nearest_peak)
    df['peak_proximity'] = 1.0 / (1 + days_to_peak)
    month = df['Date'].dt.month
    df['month_sin'] = np.sin(2 * np.pi * month / 12)
    df['month_cos'] = np.cos(2 * np.pi * month / 12)
    return df


def _get_sale_windows(years):
    windows = []
    for year in years:
        for s in SALE_SEASONS:
            try:
                start = pd.Timestamp(year=year, month=s['month'], day=s['start_day'])
            except ValueError:
                start = pd.Timestamp(year=year, month=s['month'], day=1) + pd.offsets.MonthEnd(0)
            end = start + pd.Timedelta(days=s['duration'] - 1)
            windows.append((start, end, s['profit_rank']))
    return windows


def add_sale_features(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    years = df['Date'].dt.year.unique()
    windows = _get_sale_windows(years)
    df['is_sale_season'] = 0
    df['sale_rank'] = 0
    df['days_to_next_sale'] = 999
    df['days_since_last_sale'] = 999

    for start, end, rank in windows:
        mask_in = (df['Date'] >= start) & (df['Date'] <= end)
        df.loc[mask_in, 'is_sale_season'] = 1
        df.loc[mask_in, 'sale_rank'] = rank
        mask_before = df['Date'] < start
        days_to = (start - df.loc[mask_before, 'Date']).dt.days
        df.loc[mask_before, 'days_to_next_sale'] = np.minimum(
            df.loc[mask_before, 'days_to_next_sale'], days_to
        )
        mask_after = df['Date'] > end
        days_since = (df.loc[mask_after, 'Date'] - end).dt.days
        df.loc[mask_after, 'days_since_last_sale'] = np.minimum(
            df.loc[mask_after, 'days_since_last_sale'], days_since
        )

    df.loc[df['is_sale_season'] == 1, 'days_to_next_sale'] = 0
    df.loc[df['is_sale_season'] == 1, 'days_since_last_sale'] = 0
    return df


def add_features(df):
    df = df.copy()
    df = add_seasonality_features(df)
    df = add_sale_features(df)
    return df


STATIC_FEATURES = [
    'year', 'month', 'day', 'dayofweek',
    'month_sin', 'month_cos', 'is_peak_season', 'peak_proximity',
    'is_sale_season', 'sale_rank', 'days_to_next_sale', 'days_since_last_sale',
]


In [6]:
# --- Load features: parquet (157+ features) if available, else inline CSV ---
EXCLUDE_COLS = {'Date', 'Revenue', 'COGS', 'is_train', 'Revenue_log', 'COGS_Ratio'}

if os.path.isfile(PARQUET_TRAIN) and os.path.isfile(PARQUET_TEST):
    df_train = pd.read_parquet(PARQUET_TRAIN)
    df_test  = pd.read_parquet(PARQUET_TEST)
    df_train['Date'] = pd.to_datetime(df_train['Date'])
    df_test['Date']  = pd.to_datetime(df_test['Date'])
    # Date spine may include non-trading days with NaN Revenue -- drop them
    df_train = df_train.dropna(subset=['Revenue']).reset_index(drop=True)
    print(f'Loaded parquet | train {df_train.shape} | test {df_test.shape}')
    USE_PARQUET = True
else:
    df_train = pd.read_csv(SALES_PATH, parse_dates=['Date'])
    df_test  = pd.read_csv(SUB_PATH,   parse_dates=['Date'])
    df_train = add_features(df_train)
    df_test  = add_features(df_test)
    print(f'Loaded CSV fallback | train {df_train.shape} | test {df_test.shape}')
    USE_PARQUET = False

df_train['Revenue_log'] = np.log1p(df_train['Revenue'])
df_train['COGS_Ratio']  = df_train['COGS'] / (df_train['Revenue'] + 1e-9)

STATIC_FEATURES = [c for c in df_train.columns
                   if c not in EXCLUDE_COLS and df_train[c].dtype != object]

print(f'Train: {df_train.shape} | Test: {df_test.shape}')
print(f'Static features: {len(STATIC_FEATURES)} | parquet={USE_PARQUET}')
print(df_train[['Date', 'Revenue', 'COGS']].head(3))

Loaded parquet | train (3837, 150) | test (548, 150)
Train: (3837, 152) | Test: (548, 150)
Static features: 146 | parquet=True
        Date     Revenue        COGS
0 2012-07-04  5123547.94  3982991.19
1 2012-07-05  2751773.45  2150580.23
2 2012-07-06  3054029.42  2517632.84


In [7]:
# --- CV folds (walk-forward by calendar year, same as reference notebook) ---
FOLDS = [
    ('2012-07-04', '2018-12-31', '2019-01-01', '2019-12-31'),
    ('2012-07-04', '2019-12-31', '2020-01-01', '2020-12-31'),
    ('2012-07-04', '2020-12-31', '2021-01-01', '2021-12-31'),
    ('2012-07-04', '2021-12-31', '2022-01-01', '2022-12-31'),
]

for i, (ts, te, vs, ve) in enumerate(FOLDS, 1):
    print(f'Fold {i}: train [{ts} .. {te}]  val [{vs} .. {ve}]')


Fold 1: train [2012-07-04 .. 2018-12-31]  val [2019-01-01 .. 2019-12-31]
Fold 2: train [2012-07-04 .. 2019-12-31]  val [2020-01-01 .. 2020-12-31]
Fold 3: train [2012-07-04 .. 2020-12-31]  val [2021-01-01 .. 2021-12-31]
Fold 4: train [2012-07-04 .. 2021-12-31]  val [2022-01-01 .. 2022-12-31]


In [8]:
# --- Optional Optuna (CPU); set True for full search (~minutes) ---
RUN_OPTUNA = True
N_OPTUNA_TRIALS = 30


def tune_base_models(train_df):
    if not OPTUNA_OK:
        print('[SKIP] optuna not installed')
        return None, None, None

    df = train_df.copy()
    cached_folds = []
    for train_start, train_end, val_start, val_end in FOLDS:
        tr = df[(df['Date'] >= train_start) & (df['Date'] <= train_end)].copy()
        va = df[(df['Date'] >= val_start) & (df['Date'] <= val_end)].copy()
        pr = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        pr_df = tr[['Date', 'Revenue_log']].rename(columns={'Date': 'ds', 'Revenue_log': 'y'})
        pr.fit(pr_df)
        tr['residual'] = tr['Revenue_log'].values - pr.predict(pr_df)['yhat'].values
        val_trend = pr.predict(va[['Date']].rename(columns={'Date': 'ds'}))['yhat'].values
        va['residual'] = va['Revenue_log'].values - val_trend
        cached_folds.append({
            'X_train': tr[STATIC_FEATURES], 'y_train_res': tr['residual'],
            'X_val': va[STATIC_FEATURES], 'y_val_res': va['residual'],
            'val_trend': val_trend, 'y_val_true_rev': va['Revenue'],
        })

    def objective_xgb(trial):
        params = {
            'n_estimators': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 8),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'objective': 'reg:pseudohubererror',
            'tree_method': 'hist',
            'random_state': SEED,
        }
        cv_maes = []
        for fold in cached_folds:
            model = xgb.XGBRegressor(**params, early_stopping_rounds=50)
            model.fit(
                fold['X_train'], fold['y_train_res'],
                eval_set=[(fold['X_val'], fold['y_val_res'])],
                verbose=False,
            )
            pred_rev = np.expm1(fold['val_trend'] + model.predict(fold['X_val']))
            cv_maes.append(mean_absolute_error(fold['y_val_true_rev'], pred_rev))
        return np.mean(cv_maes)

    def objective_lgb(trial):
        params = {
            'n_estimators': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'max_depth': trial.suggest_int('max_depth', 4, 8),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'objective': 'huber',
            'random_state': SEED,
            'verbose': -1,
        }
        cv_maes = []
        for fold in cached_folds:
            model = lgb.LGBMRegressor(**params)
            model.fit(
                fold['X_train'], fold['y_train_res'],
                eval_set=[(fold['X_val'], fold['y_val_res'])],
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )
            pred_rev = np.expm1(fold['val_trend'] + model.predict(fold['X_val']))
            cv_maes.append(mean_absolute_error(fold['y_val_true_rev'], pred_rev))
        return np.mean(cv_maes)

    def objective_cat(trial):
        params = {
            'iterations': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
            'loss_function': 'Huber:delta=1.5',
            'random_state': SEED,
            'verbose': False,
        }
        cv_maes = []
        for fold in cached_folds:
            model = CatBoostRegressor(**params, early_stopping_rounds=50)
            model.fit(
                fold['X_train'], fold['y_train_res'],
                eval_set=(fold['X_val'], fold['y_val_res']),
                verbose=False,
            )
            pred_rev = np.expm1(fold['val_trend'] + model.predict(fold['X_val']))
            cv_maes.append(mean_absolute_error(fold['y_val_true_rev'], pred_rev))
        return np.mean(cv_maes)

    study_x = optuna.create_study(direction='minimize')
    study_x.optimize(objective_xgb, n_trials=N_OPTUNA_TRIALS)
    study_l = optuna.create_study(direction='minimize')
    study_l.optimize(objective_lgb, n_trials=N_OPTUNA_TRIALS)
    study_c = optuna.create_study(direction='minimize')
    study_c.optimize(objective_cat, n_trials=N_OPTUNA_TRIALS)
    return study_x.best_trial.params, study_l.best_trial.params, study_c.best_trial.params


if RUN_OPTUNA:
    px, pl, pc = tune_base_models(df_train)
    print('best_xgb_params =', px)
    print('best_lgb_params =', pl)
    print('best_cat_params =', pc)
else:
    print('RUN_OPTUNA=False — using default best-params block below.')


best_xgb_params = {'learning_rate': 0.015758670237086485, 'max_depth': 5, 'subsample': 0.6786200326924503, 'colsample_bytree': 0.9064783976520754, 'min_child_weight': 1}
best_lgb_params = {'learning_rate': 0.08081790429006572, 'max_depth': 4, 'num_leaves': 46, 'subsample': 0.6618674686743957, 'colsample_bytree': 0.8683457213098044}
best_cat_params = {'learning_rate': 0.02145385122640198, 'depth': 4, 'l2_leaf_reg': 7.587453073039242}


In [10]:
# --- Fallback params (used when RUN_OPTUNA=False or optuna not installed) ---
best_xgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.07687359172082428,
    'max_depth': 4,
    'objective': 'reg:pseudohubererror',
    'tree_method': 'hist',
    'random_state': SEED,
    'subsample': 0.9640163894074474,
    'colsample_bytree': 0.9988728404218791,
    'min_child_weight': 6,
}

best_lgb_params = {
    'n_estimators': 2000,
    'learning_rate': 0.010074446630670307,
    'max_depth': 4,
    'objective': 'huber',
    'random_state': SEED,
    'verbose': -1,
    'num_leaves': 28,
    'subsample': 0.932752195699773,
    'colsample_bytree': 0.8528851999663052,
}

best_cat_params = {
    'iterations': 2000,
    'learning_rate': 0.09865972508027333,
    'depth': 4,
    'loss_function': 'Huber:delta=1.5',
    'random_state': SEED,
    'verbose': False,
    'l2_leaf_reg': 5.35578476395823,
}

# Apply Optuna results on top of fallback (Optuna only returns tuned keys,
# so update() correctly merges without losing objective/tree_method/etc.)
if RUN_OPTUNA and OPTUNA_OK:
    best_xgb_params.update(px)
    best_lgb_params.update(pl)
    best_cat_params.update(pc)
    print('Params: Optuna-tuned')
    print('  XGB:', {k: v for k, v in best_xgb_params.items() if k not in ('objective', 'tree_method', 'random_state', 'verbose')})
    print('  LGB:', {k: v for k, v in best_lgb_params.items() if k not in ('objective', 'random_state', 'verbose')})
    print('  Cat:', {k: v for k, v in best_cat_params.items() if k not in ('loss_function', 'random_state', 'verbose')})
else:
    print('Params: hardcoded fallback (RUN_OPTUNA=False or optuna missing)')

Params: Optuna-tuned
  XGB: {'n_estimators': 2000, 'learning_rate': 0.015758670237086485, 'max_depth': 5, 'subsample': 0.6786200326924503, 'colsample_bytree': 0.9064783976520754, 'min_child_weight': 1}
  LGB: {'n_estimators': 2000, 'learning_rate': 0.08081790429006572, 'max_depth': 4, 'num_leaves': 46, 'subsample': 0.6618674686743957, 'colsample_bytree': 0.8683457213098044}
  Cat: {'iterations': 2000, 'learning_rate': 0.02145385122640198, 'depth': 4, 'l2_leaf_reg': 7.587453073039242}


In [11]:
def run_prophet_stack_pipeline(train_df, test_df, xgb_p, lgb_p, cat_p, final_n_trees=600):
    """Prophet trend + Huber GBMs on residual + Huber meta; same for COGS ratio."""
    df = train_df.copy()
    df_test = test_df.copy()

    xgb_params = {
        'n_estimators': 2000, 'learning_rate': 0.03, 'max_depth': 6,
        'objective': 'reg:pseudohubererror', 'tree_method': 'hist', 'random_state': SEED,
    }
    lgb_params = {
        'n_estimators': 2000, 'learning_rate': 0.03, 'max_depth': 6,
        'objective': 'huber', 'random_state': SEED, 'verbose': -1,
    }
    cat_params = {
        'iterations': 2000, 'learning_rate': 0.03, 'depth': 6,
        'loss_function': 'Huber:delta=1.5', 'random_state': SEED, 'verbose': False,
    }
    xgb_params.update(xgb_p)
    lgb_params.update(lgb_p)
    cat_params.update(cat_p)

    oof_rev_xgb, oof_rev_lgb, oof_rev_cat = [], [], []
    oof_rat_xgb, oof_rat_lgb, oof_rat_cat = [], [], []
    oof_y_rev, oof_y_rat = [], []

    print('Walk-forward OOF...')
    for train_start, train_end, val_start, val_end in FOLDS:
        tr = df[(df['Date'] >= train_start) & (df['Date'] <= train_end)].copy()
        va = df[(df['Date'] >= val_start) & (df['Date'] <= val_end)].copy()

        pr = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        pr_df = tr[['Date', 'Revenue_log']].rename(columns={'Date': 'ds', 'Revenue_log': 'y'})
        pr.fit(pr_df)
        train_trend = pr.predict(pr_df)['yhat'].values
        val_trend = pr.predict(va[['Date']].rename(columns={'Date': 'ds'}))['yhat'].values
        tr['residual'] = tr['Revenue_log'].values - train_trend
        va['residual'] = va['Revenue_log'].values - val_trend

        X_tr, X_va = tr[STATIC_FEATURES], va[STATIC_FEATURES]

        mx = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
        mx.fit(X_tr, tr['residual'], eval_set=[(X_va, va['residual'])], verbose=False)
        ml = lgb.LGBMRegressor(**lgb_params)
        ml.fit(
            X_tr, tr['residual'],
            eval_set=[(X_va, va['residual'])],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        mc = CatBoostRegressor(**cat_params, early_stopping_rounds=50)
        mc.fit(X_tr, tr['residual'], eval_set=(X_va, va['residual']), verbose=False)

        oof_rev_xgb.extend(val_trend + mx.predict(X_va))
        oof_rev_lgb.extend(val_trend + ml.predict(X_va))
        oof_rev_cat.extend(val_trend + mc.predict(X_va))
        oof_y_rev.extend(va['Revenue_log'].values)

        rx = xgb.XGBRegressor(**xgb_params, early_stopping_rounds=50)
        rx.fit(X_tr, tr['COGS_Ratio'], eval_set=[(X_va, va['COGS_Ratio'])], verbose=False)
        rl = lgb.LGBMRegressor(**lgb_params)
        rl.fit(
            X_tr, tr['COGS_Ratio'],
            eval_set=[(X_va, va['COGS_Ratio'])],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )
        rc = CatBoostRegressor(**cat_params, early_stopping_rounds=50)
        rc.fit(X_tr, tr['COGS_Ratio'], eval_set=(X_va, va['COGS_Ratio']), verbose=False)

        oof_rat_xgb.extend(rx.predict(X_va))
        oof_rat_lgb.extend(rl.predict(X_va))
        oof_rat_cat.extend(rc.predict(X_va))
        oof_y_rat.extend(va['COGS_Ratio'].values)

    X_meta_rev = np.column_stack([oof_rev_xgb, oof_rev_lgb, oof_rev_cat])
    huber_grid = {'epsilon': [1.2, 1.25, 1.35, 1.4, 1.45, 1.5], 'alpha': [0.0001, 0.001, 0.01, 0.1]}
    meta_rev_search = GridSearchCV(
        HuberRegressor(fit_intercept=False), huber_grid, cv=5, scoring='neg_mean_absolute_error',
    )
    meta_rev_search.fit(X_meta_rev, oof_y_rev)
    meta_rev = meta_rev_search.best_estimator_

    X_meta_rat = np.column_stack([oof_rat_xgb, oof_rat_lgb, oof_rat_cat])
    huber_rat = {'epsilon': [1.01, 1.05, 1.1, 1.2], 'alpha': [0.0001, 0.001, 0.01, 0.1]}
    meta_rat_search = GridSearchCV(
        HuberRegressor(fit_intercept=False), huber_rat, cv=5, scoring='neg_mean_absolute_error',
    )
    meta_rat_search.fit(X_meta_rat, oof_y_rat)
    meta_rat = meta_rat_search.best_estimator_

    oof_pred_log = meta_rev.predict(X_meta_rev)
    print('Meta Revenue coef (XGB, LGB, Cat):', meta_rev.coef_)
    print('OOF MAE Revenue (level):', mean_absolute_error(np.expm1(oof_y_rev), np.expm1(oof_pred_log)))
    print('Meta Ratio coef:', meta_rat.coef_)
    print('OOF MAE COGS_Ratio:', mean_absolute_error(oof_y_rat, meta_rat.predict(X_meta_rat)))

    xp = {k: v for k, v in xgb_params.items()}
    lp = {k: v for k, v in lgb_params.items()}
    cp = {k: v for k, v in cat_params.items()}
    xp['n_estimators'] = final_n_trees
    lp['n_estimators'] = final_n_trees
    cp['iterations'] = final_n_trees

    final_pr = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    full_prophet_df = df[['Date', 'Revenue_log']].rename(columns={'Date': 'ds', 'Revenue_log': 'y'})
    final_pr.fit(full_prophet_df)
    df['residual'] = df['Revenue_log'].values - final_pr.predict(full_prophet_df)['yhat'].values

    f_rev_x = xgb.XGBRegressor(**xp)
    f_rev_l = lgb.LGBMRegressor(**lp)
    f_rev_c = CatBoostRegressor(**cp)
    f_rev_x.fit(df[STATIC_FEATURES], df['residual'], verbose=False)
    f_rev_l.fit(df[STATIC_FEATURES], df['residual'])
    f_rev_c.fit(df[STATIC_FEATURES], df['residual'], verbose=False)

    f_rat_x = xgb.XGBRegressor(**xp)
    f_rat_l = lgb.LGBMRegressor(**lp)
    f_rat_c = CatBoostRegressor(**cp)
    f_rat_x.fit(df[STATIC_FEATURES], df['COGS_Ratio'], verbose=False)
    f_rat_l.fit(df[STATIC_FEATURES], df['COGS_Ratio'])
    f_rat_c.fit(df[STATIC_FEATURES], df['COGS_Ratio'], verbose=False)

    test_trend = final_pr.predict(df_test[['Date']].rename(columns={'Date': 'ds'}))['yhat'].values
    Xt = df_test[STATIC_FEATURES]
    p_log_x = test_trend + f_rev_x.predict(Xt)
    p_log_l = test_trend + f_rev_l.predict(Xt)
    p_log_c = test_trend + f_rev_c.predict(Xt)
    X_te_rev = np.column_stack([p_log_x, p_log_l, p_log_c])
    df_test = df_test.copy()
    df_test['Revenue'] = np.clip(np.expm1(meta_rev.predict(X_te_rev)), 0, None)

    pr_x = f_rat_x.predict(Xt)
    pr_l = f_rat_l.predict(Xt)
    pr_c = f_rat_c.predict(Xt)
    X_te_rat = np.column_stack([pr_x, pr_l, pr_c])
    rat_hat = meta_rat.predict(X_te_rat)
    df_test['COGS'] = np.clip(df_test['Revenue'].values * rat_hat, 0, None)

    out = {
        'meta_rev': meta_rev,
        'meta_rat': meta_rat,
        'prophet': final_pr,
        'f_rev_xgb': f_rev_x,
        'f_rev_lgb': f_rev_l,
        'f_rev_cat': f_rev_c,
        'f_rat_xgb': f_rat_x,
        'f_rat_lgb': f_rat_l,
        'f_rat_cat': f_rat_c,
        'df_train': df,
        'df_test': df_test,
    }
    return out


result = run_prophet_stack_pipeline(df_train, df_test, best_xgb_params, best_lgb_params, best_cat_params)

submission = result['df_test'][['Date', 'Revenue', 'COGS']].sort_values('Date')
submission.to_csv(f'{SUB_DIR}/submission_modeling_v2.csv', index=False)
print('Saved:', f'{SUB_DIR}/submission_modeling_v2.csv')


Walk-forward OOF...
Meta Revenue coef (XGB, LGB, Cat): [-0.00331962  0.12208271  0.88687961]
OOF MAE Revenue (level): 674307.1529535969
Meta Ratio coef: [ 0.72401716  0.42615049 -0.14947578]
OOF MAE COGS_Ratio: 0.02098729779044801
Saved: ../submissions/submission_modeling_v2.csv


In [12]:
# --- Post-processing: per-(m,d) median shrinkage ---
# If parquet features include rev_mday_median_all, blend ML prediction with the
# per-(month,day) historical median shrunk 7% toward the overall median.
# This reduces variance at spike days and is the core trick of the 674k notebook.
# BLEND_W=1 → pure ML; BLEND_W=0 → pure shrunk median. Tune on val 2022.

submission = result['df_test'][['Date', 'Revenue', 'COGS']].sort_values('Date').reset_index(drop=True)

if 'rev_mday_median_all' in result['df_test'].columns:
    df_te_sorted = result['df_test'].sort_values('Date').reset_index(drop=True)
    q50 = df_te_sorted['rev_mday_median_all'].values.astype(float).copy()
    q50[np.isnan(q50)] = np.nanmedian(q50)

    overall_median = float(np.median(q50))
    SHRINK = 0.07
    shrunk = np.clip(q50 - SHRINK * (q50 - overall_median), 0, None)

    BLEND_W = 0.5   # weight on ML; 1-BLEND_W on shrunk median
    blended_rev = np.clip(BLEND_W * submission['Revenue'].values + (1 - BLEND_W) * shrunk, 0, None)
    ml_ratio    = submission['COGS'].values / (submission['Revenue'].values + 1e-9)
    blended_cogs = np.clip(blended_rev * ml_ratio, 0, blended_rev)

    submission_blended = pd.DataFrame({
        'Date': submission['Date'].values,
        'Revenue': blended_rev.round(2),
        'COGS': blended_cogs.round(2),
    })
    submission_blended.to_csv(f'{SUB_DIR}/submission_v2_blended.csv', index=False)

    print(f'Saved: {SUB_DIR}/submission_v2_blended.csv')
    print(f'  Pure ML Rev mean  : {submission["Revenue"].mean():>12,.0f}')
    print(f'  Shrunk q50 mean   : {shrunk.mean():>12,.0f}  (SHRINK={SHRINK})')
    print(f'  Blended Rev mean  : {blended_rev.mean():>12,.0f}  (BLEND_W={BLEND_W})')
    print()
    print('Tune BLEND_W: try 0.3, 0.5, 0.7 on val-2022 MAE; re-run run_prophet_stack_pipeline.')
else:
    print('[SKIP] rev_mday_median_all not found — re-run notebooks/30-feature-engineering.ipynb first')

submission.to_csv(f'{SUB_DIR}/submission_modeling_v2.csv', index=False)
print(f'Saved ML-only: {SUB_DIR}/submission_modeling_v2.csv')

Saved: ../submissions/submission_v2_blended.csv
  Pure ML Rev mean  :    3,464,891
  Shrunk q50 mean   :    4,457,988  (SHRINK=0.07)
  Blended Rev mean  :    3,961,440  (BLEND_W=0.5)

Tune BLEND_W: try 0.3, 0.5, 0.7 on val-2022 MAE; re-run run_prophet_stack_pipeline.
Saved ML-only: ../submissions/submission_modeling_v2.csv


In [13]:
# --- Feature importance (gain) on residual models ---
fi_x = pd.DataFrame({'feature': STATIC_FEATURES, 'gain': result['f_rev_xgb'].feature_importances_})
fi_l = pd.DataFrame({'feature': STATIC_FEATURES, 'gain': result['f_rev_lgb'].booster_.feature_importance(importance_type='gain')})
fi_c = pd.DataFrame({'feature': STATIC_FEATURES, 'gain': result['f_rev_cat'].get_feature_importance()})

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, fi, title, color in zip(
    axes,
    (fi_x, fi_l, fi_c),
    ('XGB residual', 'LGB residual', 'Cat residual'),
    ('#2563EB', '#16A34A', '#CA8A04'),
):
    fi = fi.sort_values('gain', ascending=True).tail(20)
    ax.barh(fi['feature'], fi['gain'], color=color)
    ax.set_title(title)

plt.tight_layout()
plt.savefig(f'{CHART_DIR}/feature_importance_modeling_v2_residual.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- SHAP (TreeExplainer) on residual targets ---
SHAP_BG_N = 400
SHAP_EX_N = 600

if not SHAP_OK:
    print('[SKIP] pip install shap')
else:
    rng = np.random.RandomState(SEED)
    df_tr = result['df_train']
    n_train = len(df_tr)
    bg_n = min(SHAP_BG_N, n_train)
    ex_n = min(SHAP_EX_N, n_train)
    bg_idx = rng.choice(n_train, size=bg_n, replace=False)
    ex_idx = rng.choice(n_train, size=ex_n, replace=False)
    X_bg = df_tr[STATIC_FEATURES].iloc[bg_idx]
    X_ex = df_tr[STATIC_FEATURES].iloc[ex_idx]

    def save_shap(model, X_bg_loc, X_ex_loc, path_png, title):
        # Removed check_additivity=False as it's not supported by the current shap version
        expl = shap.TreeExplainer(model, X_bg_loc)
        sv = expl.shap_values(X_ex_loc)
        plt.figure(figsize=(10, 8))
        shap.summary_plot(sv, X_ex_loc, feature_names=STATIC_FEATURES, show=False, max_display=22)
        plt.title(title, fontsize=11)
        plt.tight_layout()
        plt.savefig(path_png, dpi=150, bbox_inches='tight')
        plt.close()

    print('SHAP XGB (log residual)...')
    save_shap(result['f_rev_xgb'], X_bg, X_ex, f'{CHART_DIR}/shap_v2_xgb_residual.png', 'SHAP v2 XGB log-residual')
    print('SHAP LGB ...')
    save_shap(result['f_rev_lgb'], X_bg, X_ex, f'{CHART_DIR}/shap_v2_lgb_residual.png', 'SHAP v2 LGB log-residual')
    print('SHAP Cat ...')
    save_shap(result['f_rev_cat'], X_bg, X_ex, f'{CHART_DIR}/shap_v2_cat_residual.png', 'SHAP v2 Cat log-residual')

    X_bg_r = X_bg.copy()
    X_ex_r = X_ex.copy()
    print('SHAP XGB COGS_Ratio...')
    save_shap(result['f_rat_xgb'], X_bg_r, X_ex_r, f'{CHART_DIR}/shap_v2_xgb_cogs_ratio.png', 'SHAP v2 XGB COGS ratio')
    print('Saved charts under', CHART_DIR)


In [17]:
# --- Forecast plot: train tail vs test prediction ---
train_tail = df_train.sort_values('Date').tail(240)
test_pred = result['df_test'].sort_values('Date')

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(train_tail['Date'], train_tail['Revenue'], color='gray', alpha=0.55, lw=1, label='Train actual')
ax.plot(test_pred['Date'], test_pred['Revenue'], color='#7C3AED', lw=1.5, label='v2 Predicted Revenue')
ax.axvline(x=test_pred['Date'].min(), color='red', ls='--', alpha=0.45)
ax.set_title('40-modeling-v2: Prophet + residual stack (Revenue)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend()
plt.tight_layout()
plt.savefig(f'{CHART_DIR}/forecast_modeling_v2.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- Summary ---
print('=' * 72)
print('40-modeling-v2 SUMMARY')
print('=' * 72)
print('  Data source      :', SALES_PATH)
print('  Static features  :', len(STATIC_FEATURES))
print('  Pipeline         : Prophet(log Revenue) + XGB/LGB/Cat Huber on residual')
print('  Stack            : HuberRegressor OOF meta (3 bases)')
print('  COGS             : meta-stack on COGS/Revenue (same features)')
print('  Submission       :', f'{SUB_DIR}/submission_modeling_v2.csv')
print('  Test Revenue mean:', submission['Revenue'].mean():,.0f)
print('  Test COGS mean   :', submission['COGS'].mean():,.0f)
print('=' * 72)
